# SmartRoute-OSM: Algoritmo de Dijkstra

Neste segundo notebook do pipeline, carregaremos o arquivo `.graphml` salvo na etapa anterior. Em seguida, usaremos o **Algoritmo de Dijkstra** (implementado em `routing/algorithms.py`) para encontrar o menor caminho (com base na distância em metros) entre dois pontos na malha viária de Quixadá.

In [1]:
import os
import sys
import time
import osmnx as ox
import pandas as pd

# Adicionar pasta do projeto ao path para importar módulo de roteamento
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'routing'))
from algorithms import dijkstra

# Carregar o grafo pré-processado
data_path = "../data/quixada_drive.graphml"

if os.path.exists(data_path):
    G = ox.load_graphml(data_path)
    print(f"Grafo carregado com sucesso! Nós: {len(G.nodes)}, Arestas: {len(G.edges)}")
else:
    raise FileNotFoundError("Arquivo 'quixada_drive.graphml' não encontrado. Execute o Notebook 01 primeiro.")

Grafo carregado com sucesso! Nós: 3647, Arestas: 9762


## 1. Definição do Par Origem-Destino
Escolhemos coordenadas de latitude e longitude na cidade e identificamos os nós da malha viária mais próximos a elas.

In [2]:
# Coordenadas fixas de exemplo em Quixadá
origem_coords = (-4.9685, -39.0161)
destino_coords = (-4.9780, -39.0050)

# Mapear coordenadas para os nós do grafo
origem_node = ox.distance.nearest_nodes(G, X=origem_coords[1], Y=origem_coords[0])
destino_node = ox.distance.nearest_nodes(G, X=destino_coords[1], Y=destino_coords[0])

print(f"ID Nó Origem: {origem_node}")
print(f"ID Nó Destino: {destino_node}")

ID Nó Origem: 252615233
ID Nó Destino: 4829461035


## 2. Execução e Medição de Desempenho
Executamos o algoritmo medindo o tempo total de processamento em milissegundos.

In [3]:
start_time = time.time()
dijkstra_path, dijkstra_dist, dijkstra_visited = dijkstra(G, origem_node, destino_node, weight_attribute='length')
execution_time_ms = (time.time() - start_time) * 1000

print(f"=== RESULTADOS DIJKSTRA ===")
print(f"Distância Total: {dijkstra_dist:.2f} metros")
print(f"Nós Visitados: {dijkstra_visited}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

=== RESULTADOS DIJKSTRA ===
Distância Total: 1890.27 metros
Nós Visitados: 961
Tempo de Execução: 4.71 ms


## 4. Salvando Resultados Intermediários
Salvamos as métricas em formato CSV para consolidar no notebook final de benchmark (`04_benchmark_and_folium_map.ipynb`).

In [4]:
import json

metrics_data = {
    'algorithm': ['Dijkstra'],
    'distance_m': [dijkstra_dist],
    'visited_nodes': [dijkstra_visited],
    'execution_time_ms': [execution_time_ms],
    'origem_node': [origem_node],
    'destino_node': [destino_node]
}

df_dijkstra = pd.DataFrame(metrics_data)
df_dijkstra.to_csv("../data/dijkstra_metrics.csv", index=False)

# Salvar o caminho para reusar no mapa
with open("../data/dijkstra_path.json", "w") as f:
    json.dump(dijkstra_path, f)

print("Métricas e caminho salvos na pasta '../data/' com sucesso!")

Métricas e caminho salvos na pasta '../data/' com sucesso!
